# Mindset generator explorer

interactive notebook for browsing and previewing all 33 mindset-vision generators.

```bash
pip install -e ".[notebook]"  # from repo root
```

In [ ]:
import csv
import io
import os
import sys
import base64
import tempfile
import shutil
import dataclasses
from pathlib import Path
from dataclasses import fields, asdict

if not Path("pyproject.toml").exists() and Path("../pyproject.toml").exists():
    os.chdir("..")

import yaml
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from PIL import Image as PILImage

from mindset.cli import _load_registry

REGISTRY = _load_registry()
PY = sys.executable
print(f"loaded {len(REGISTRY)} generators")


def pil_to_data_uri(img, max_size=224):
    """convert a PIL image to an inline data URI."""
    img.thumbnail((max_size, max_size), PILImage.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"


def render_grid_html(images, labels, title=""):
    """render a list of PIL images as an HTML flex grid."""
    cells = "".join(
        f'<div style="text-align:center;margin:4px">'
        f'<img src="{pil_to_data_uri(img)}" style="max-width:180px;border:1px solid #ddd"><br>'
        f'<small style="color:#555">{label}</small></div>'
        for img, label in zip(images, labels)
    )
    return HTML(
        f'<div style="margin:8px 0"><b>{title}</b></div>'
        f'<div style="display:flex;flex-wrap:wrap;gap:4px">{cells}</div>'
    )

## Interactive explorer

select a generator, adjust parameters via sliders, and preview. export config to yaml for CLI use.

In [ ]:
class GeneratorExplorer:
    """interactive explorer for mindset-vision generators."""

    SKIP_FIELDS = {"output_folder", "behaviour_if_present"}
    PREVIEW_SAMPLES = 3

    def __init__(self, registry):
        """build the explorer UI."""
        self.registry = registry
        self.param_widgets = {}
        self.current_name = None

        by_cat = {}
        for name, info in sorted(registry.items()):
            by_cat.setdefault(info["category"], []).append(name)

        options = []
        for cat, names in sorted(by_cat.items()):
            for name in names:
                options.append((f"[{cat}] {name}", name))

        style = {"description_width": "initial"}
        self.selector = widgets.Dropdown(
            options=options, description="generator:",
            style=style, layout=widgets.Layout(width="420px"),
        )
        self.selector.observe(self._on_select, names="value")
        self.desc_html = widgets.HTML()
        self.param_box = widgets.VBox()
        self.btn = widgets.Button(description="generate preview", button_style="primary", icon="play")
        self.btn.on_click(self._on_generate)
        self.export_btn = widgets.Button(description="export yaml config", button_style="", icon="download")
        self.export_btn.on_click(self._on_export)
        self.output = widgets.Output()
        self._on_select(None)

    def show(self):
        """display the explorer widget."""
        return widgets.VBox([
            self.selector,
            self.desc_html,
            self.param_box,
            widgets.HBox([self.btn, self.export_btn]),
            self.output,
        ])

    def _widget_from_field(self, fld):
        """create an ipywidget from a dataclass field."""
        meta = dict(fld.metadata)
        label = meta.get("label", fld.name.replace("_", " "))
        default = fld.default if fld.default is not dataclasses.MISSING else fld.default_factory()
        style = {"description_width": "180px"}
        layout = widgets.Layout(width="520px")
        has_range = "min" in meta and "max" in meta

        if fld.type == int and "sample" in fld.name:
            default = min(default, self.PREVIEW_SAMPLES)
            meta["max"] = min(meta.get("max", 100), 20)

        type_name = fld.type.__name__ if hasattr(fld.type, "__name__") else str(fld.type)

        match type_name:
            case "bool":
                return widgets.Checkbox(value=default, description=label, indent=False, layout=layout)
            case "int" if has_range:
                return widgets.IntSlider(value=default, min=meta["min"], max=meta["max"], step=meta.get("step", 1), description=label, style=style, layout=layout)
            case "int":
                return widgets.IntText(value=default, description=label, style=style, layout=layout)
            case "float" if has_range:
                return widgets.FloatSlider(value=default, min=meta["min"], max=meta["max"], step=meta.get("step", 0.01), description=label, style=style, layout=layout)
            case "float":
                return widgets.FloatText(value=default, description=label, style=style, layout=layout)
            case "str" if "choices" in meta:
                return widgets.Dropdown(options=meta["choices"], value=default, description=label, style=style, layout=layout)
            case "list" if ("color" in fld.name or "colour" in fld.name) and isinstance(default, list) and len(default) >= 3:
                hex_val = "#{:02x}{:02x}{:02x}".format(*default[:3])
                return widgets.ColorPicker(value=hex_val, description=label, style=style, layout=layout)
            case "list" if has_range:
                val = default[0] if isinstance(default, list) else default
                return widgets.IntSlider(value=val, min=meta["min"], max=meta["max"], step=meta.get("step", 16), description=label, style=style, layout=layout)
            case "list" if isinstance(default, list) and len(default) == 2 and all(isinstance(v, (int, float)) for v in default):
                lo, hi = default
                span = abs(hi - lo) or 1
                return widgets.FloatRangeSlider(
                    value=default, min=lo - span, max=hi + span,
                    step=round(span / 20, 4) or 0.01,
                    description=label, style=style, layout=layout,
                )
            case _:
                return widgets.Text(value=str(default), description=label, style=style, layout=layout)

    def _on_select(self, _change):
        """rebuild parameter widgets and description when generator changes."""
        self.current_name = self.selector.value
        info = self.registry[self.current_name]
        config_cls = info["config_cls"]

        doc = (info["func"].__doc__ or config_cls.__doc__ or "").strip()
        self.desc_html.value = f'<div style="color:#555;font-size:13px;margin:4px 0 8px">{doc}</div>'

        self.param_widgets = {}
        widget_list = []
        for fld in fields(config_cls):
            if fld.name in self.SKIP_FIELDS:
                continue
            w = self._widget_from_field(fld)
            if w is not None:
                self.param_widgets[fld.name] = (fld, w)
                widget_list.append(w)
        self.param_box.children = widget_list

    def _extract_kwargs(self):
        """read current widget values into a kwargs dict."""
        kwargs = {}
        for name, (fld, widget) in self.param_widgets.items():
            val = widget.value
            if isinstance(widget, widgets.ColorPicker):
                h = val.lstrip("#")
                val = [int(h[i:i+2], 16) for i in (0, 2, 4)]
            elif isinstance(widget, widgets.FloatRangeSlider):
                val = list(val)
            elif fld.type == list and "size" in fld.name:
                val = [val, val]
            kwargs[name] = val
        return kwargs

    def _on_export(self, _btn):
        """export current slider values as a yaml config file."""
        with self.output:
            clear_output(wait=True)
            kwargs = self._extract_kwargs()
            config_cls = self.registry[self.current_name]["config_cls"]
            config = config_cls(**kwargs)
            out_path = Path(f"{self.current_name}.yaml")
            with open(out_path, "w") as fh:
                yaml.dump(asdict(config), fh, default_flow_style=False, sort_keys=False)
            cmd = f"mindset generate {self.current_name} --config {out_path.resolve()}"
            display(HTML(
                f'<div style="background:#1e1e1e;color:#d4d4d4;padding:10px;border-radius:4px;font-family:monospace;font-size:12px">{cmd}</div>'
                f'<p style="font-size:11px;color:#888">config saved to {out_path.resolve()}</p>'
            ))

    def _on_generate(self, _btn):
        """generate preview images and display as HTML grid."""
        with self.output:
            clear_output(wait=True)
            self.btn.disabled = True
            self.btn.description = "generating..."

            tmp = Path(tempfile.mkdtemp(prefix="mindset_preview_"))
            kwargs = self._extract_kwargs()
            kwargs["output_folder"] = str(tmp / self.current_name)
            kwargs["behaviour_if_present"] = "overwrite"

            self.registry[self.current_name]["func"](**kwargs)

            self.btn.disabled = False
            self.btn.description = "generate preview"

            # show annotation schema
            csv_path = Path(kwargs["output_folder"]) / "annotation.csv"
            if csv_path.exists():
                with open(csv_path) as f:
                    reader = csv.reader(f)
                    headers = next(reader)
                    first_row = next(reader, None)
                hdr_html = "".join(f'<th style="padding:2px 6px;background:#eee;font-size:11px">{h}</th>' for h in headers)
                row_html = ""
                if first_row:
                    row_html = "<tr>" + "".join(f'<td style="padding:2px 6px;font-size:11px">{v}</td>' for v in first_row) + "</tr>"
                display(HTML(f'<table style="border-collapse:collapse;margin:4px 0"><tr>{hdr_html}</tr>{row_html}</table>'))

            # collect one image per condition
            all_pngs = sorted(Path(kwargs["output_folder"]).rglob("*.png"))
            conditions = {}
            for p in all_pngs:
                cond = p.parent.name
                if cond not in conditions:
                    conditions[cond] = p

            images = [PILImage.open(p) for p in list(conditions.values())[:12]]
            labels = list(conditions.keys())[:12]

            if images:
                display(render_grid_html(images, labels, f"{self.current_name} ({len(all_pngs)} images, {len(conditions)} conditions)"))
            else:
                print("no images generated")

            shutil.rmtree(tmp, ignore_errors=True)

### Generation workflow

1. use the explorer above to browse generators and adjust parameters
2. click "export yaml config" to save your settings
3. generate the full dataset via CLI:

```
mindset generate ebbinghaus --config ebbinghaus.yaml -o data/ebbinghaus
```

or dump defaults first: `mindset generate ebbinghaus --save-config`

In [ ]:
explorer = GeneratorExplorer(REGISTRY)
display(explorer.show())

## Generator catalog (optional)

run cells below to see one sample per condition for each generator. these are slow to run but useful for a full visual sweep.

In [ ]:
def preview(name, path=Path(".preview")):
    """display generated images and annotation schema."""
    out = path / name
    conditions = {}
    for p in sorted(out.rglob("*.png")):
        conditions.setdefault(p.parent.name, p)
    images = [PILImage.open(p) for p in list(conditions.values())[:10]]
    labels = list(conditions.keys())[:10]
    if images:
        display(render_grid_html(images, labels, name))
    csv_path = out / "annotation.csv"
    if csv_path.exists():
        with open(csv_path) as f:
            reader = csv.reader(f)
            headers = next(reader)
            row = next(reader, None)
        hdr = "".join(f'<th style="padding:1px 6px;background:#f5f5f5;font-size:10px">{h}</th>' for h in headers)
        vals = ""
        if row:
            vals = "<tr>" + "".join(f'<td style="padding:1px 6px;font-size:10px">{v[:20]}</td>' for v in row) + "</tr>"
        display(HTML(f'<table style="border-collapse:collapse;margin:4px 0"><tr>{hdr}</tr>{vals}</table>'))

### Low and mid-level vision

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate amodal_completion -o .preview/amodal_completion --samples 1
preview("amodal_completion")
# !mindset generate amodal_completion -o data/amodal_completion --num-samples 5 --circle-color 255 255 255 --square-color 0 0 0

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate decomposition -o .preview/decomposition --samples 1
preview("decomposition")
# !mindset generate decomposition -o data/decomposition --moving-distance 60 --shape-color 255 255 255 --number-unfamiliar-shapes 5

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate depth_drawings -o .preview/depth_drawings --samples 1
preview("depth_drawings")
# !mindset generate depth_drawings -o data/depth_drawings --object-longest-side 200 --input-folder mindset/assets/enns_rensink_1991/pngs

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate emergent_features -o .preview/emergent_features --samples 1
preview("emergent_features")
# !mindset generate emergent_features -o data/emergent_features --num-samples 1000

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate nap_vs_mp_2d -o .preview/nap_vs_mp_2d --samples 1
preview("nap_vs_mp_2d")
# !mindset generate nap_vs_mp_2d -o data/nap_vs_mp_2d --object-longest-side 200 --input-folder mindset/assets/kubilius_2017/pngs

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate nap_vs_mp_3d -o .preview/nap_vs_mp_3d --samples 1
preview("nap_vs_mp_3d")
# !mindset generate nap_vs_mp_3d -o data/nap_vs_mp_3d --object-longest-side 200 --shape-folder mindset/assets/amir_geons/cropped/NAPvsMP

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate relational_vs_coordinate -o .preview/relational_vs_coordinate --samples 1
preview("relational_vs_coordinate")
# !mindset generate relational_vs_coordinate -o data/relational_vs_coordinate --object-longest-side 200 --input-folder mindset/assets/hummel_stankiewicz_1996/pngs

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate uncrowding -o .preview/uncrowding --samples 1
preview("uncrowding")
# !mindset generate uncrowding -o data/uncrowding --num-samples-vernier-inside 100 --num-samples-vernier-outside 100 --random-size

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate weber_law -o .preview/weber_law --samples 1
preview("weber_law")
# !mindset generate weber_law -o data/weber_law --num-samples-per-condition 50 --max-line-length 50 --min-line-length 5 --interval-line-length 1 --min-grayscale 50 --max-grayscale 255 --interval-grayscale 20 --width 2

### Shape and object recognition

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate dotted_linedrawings -o .preview/dotted_linedrawings --samples 1
preview("dotted_linedrawings")
# !mindset generate dotted_linedrawings -o data/dotted_linedrawings --object-longest-side 200 --linedrawing-input-folder mindset/assets/baker_2018_linedrawings/cropped/ --dot-distance 5 --dot-size 1

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate embedded_figures -o .preview/embedded_figures --samples 1
preview("embedded_figures")
# !mindset generate embedded_figures -o data/embedded_figures --num-samples 100 --shape-size 45

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate global_change -o .preview/global_change --samples 1
preview("global_change")
# !mindset generate global_change -o data/global_change --object-longest-side 120 --image-input-folder mindset/assets/baker_2018_linedrawings/cropped/ --convert-to-silhouettes 0

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate global_change_baker2022 -o .preview/global_change_baker2022 --samples 1
preview("global_change_baker2022")
# !mindset generate global_change_baker2022 -o data/global_change_baker2022 --object-longest-side 200

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate leuven_embedded -o .preview/leuven_embedded --samples 1
preview("leuven_embedded")
# !mindset generate leuven_embedded -o data/leuven_embedded

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate linedrawings -o .preview/linedrawings --samples 1
preview("linedrawings")
# !mindset generate linedrawings -o data/linedrawings --object-longest-side 200 --linedrawing-input-folder mindset/assets/baker_2018_linedrawings/cropped/

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate same_different -o .preview/same_different --samples 1
preview("same_different")
# !mindset generate same_different -o data/same_different --num-samples 5000 --size-shapes 20 --type-dataset all

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate segmented_images -o .preview/segmented_images --samples 1
preview("segmented_images")
# !mindset generate segmented_images -o data/segmented_images --linedrawing-input-folder mindset/assets/baker_2018_linedrawings/cropped --object-longest-side 200 --grid-degree 45 --grid-size 8 --grid-thickness 4

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate silhouettes -o .preview/silhouettes --samples 1
preview("silhouettes")
# !mindset generate silhouettes -o data/silhouettes --object-longest-side 200 --image-input-folder mindset/assets/baker_2018_linedrawings/cropped/ --input-image-type linedrawings

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate texturized_blobs -o .preview/texturized_blobs --samples 1
preview("texturized_blobs")
# !mindset generate texturized_blobs -o data/texturized_blobs --num-samples-per-blob 5 --num-blobs 10 --object-longest-side 200 --background-char   --foreground-char random --font-size 15 20

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate texturized_chars -o .preview/texturized_chars --samples 1
preview("texturized_chars")
# !mindset generate texturized_chars -o data/texturized_chars --linedrawing-input-folder mindset/assets/baker_2018_linedrawings/cropped/ --num-samples 500 --object-longest-side 200 --background-char   --foreground-char random --font-size 15 20

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate texturized_lines -o .preview/texturized_lines --samples 1
preview("texturized_lines")
# !mindset generate texturized_lines -o data/texturized_lines --linedrawing-input-folder mindset/assets/baker_2018_linedrawings/cropped/ --num-samples 500 --object-longest-side 200 --density 1.8 --texturize-foreground

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate transformations_2d -o .preview/transformations_2d --samples 1
preview("transformations_2d")
# !mindset generate transformations_2d -o data/transformations_2d --input-folder mindset/assets/baker_2018_linedrawings/cropped/ --object-longest-side 200 --translation-x -0.2 0.2 --translation-y -0.2 0.2 --scale 0.5 0.9 --rotation 0 360 --num-samples 50

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate viewpoint_invariance -o .preview/viewpoint_invariance --samples 1
preview("viewpoint_invariance")
# !mindset generate viewpoint_invariance -o data/viewpoint_invariance --eth-80-folder mindset/assets/ETH_80 --object-longest-side 200 --azimuth-lim 0 365 --inclination-lim 30 90

### Visual illusions

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate adelson_checkerboard -o .preview/adelson_checkerboard --samples 1
preview("adelson_checkerboard")
# !mindset generate adelson_checkerboard -o data/adelson_checkerboard --steps-arrow 5 --grayscale-background 0

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate ebbinghaus -o .preview/ebbinghaus --samples 1
preview("ebbinghaus")
# !mindset generate ebbinghaus -o data/ebbinghaus --num-samples-scrambled 5000 --num-samples-illusory 50

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate grayscale_shapes -o .preview/grayscale_shapes --samples 1
preview("grayscale_shapes")
# !mindset generate grayscale_shapes -o data/grayscale_shapes --num-samples 5000 --arrow-size 1

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate jastrow -o .preview/jastrow --samples 1
preview("jastrow")
# !mindset generate jastrow -o data/jastrow --num-samples-illusory 50 --num-samples-random 1000 --num-samples-aligned 50 --num-samples-random-same-size 50

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate lightness_contrast -o .preview/lightness_contrast --samples 1
preview("lightness_contrast")
# !mindset generate lightness_contrast -o data/lightness_contrast --steps-arrow 30 --square-color 200 --steps-bg-color 20

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate muller_lyer -o .preview/muller_lyer --samples 1
preview("muller_lyer")
# !mindset generate muller_lyer -o data/muller_lyer --num-samples-scrambled 5000 --num-samples-illusory 500

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate ponzo -o .preview/ponzo --samples 1
preview("ponzo")
# !mindset generate ponzo -o data/ponzo --num-samples-scrambled 5000 --num-samples-illusory 500 --num-rail-lines 5

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate thatcher_face -o .preview/thatcher_face --samples 1
preview("thatcher_face")
# !mindset generate thatcher_face -o data/thatcher_face --face-folder mindset/assets/celebA_sample/normal

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate thatcher_words -o .preview/thatcher_words --samples 1
preview("thatcher_words")
# !mindset generate thatcher_words -o data/thatcher_words --jittery 0.04 --num-words 100 --num-samples-per-word 5 --num-letters-per-word 5 9 --num-letters-to-rotate 2 --size-fonts 18 35

In [ ]:
!TQDM_DISABLE=1 {PY} -m mindset.cli generate tilt -o .preview/tilt --samples 1
preview("tilt")
# !mindset generate tilt -o data/tilt --num-samples-only-center 1000 --num-samples-only-context 1000 --num-samples-center-context 1000